## Bài tập về nhà  
Giao diện cho bài toán máy hút bụi với các thuật toán tìm kiếm:  
Bổ sung thêm cho buổi 7:  
- Greedy Search  
- A*  
  
Ở buổi 5 đã xây dựng các thuật toán:
- BFS cách tiếp cận 1  
- BFS cách tiếp cận 2  
- DFS cách tiếp cận 1  
- DFS cách tiếp cận 2  

Ở buổi 6 đã xây dựng các thuật toán:  
- IDS cách tiếp cận 1  
- IDS cách tiếp cận 2  
- UCS  

In [72]:
# Class định nghĩa các thông tin của trạng thái
class Node:
    def __init__(self, state, parent, action, path_cost):
        self.state = state
        self.parent = parent
        self.action = action
        self.path_cost = path_cost

In [73]:
# Trả về list là chuỗi đường đi đến node truyền vào
def get_solution(node):
    path = []

    while node != None:
        path.append(node)
        node = node.parent

    path.reverse()
    return path

In [74]:
# So sánh trạng thái với trạng thái đích (Không còn bụi)
def compare_state(state, goal_state):
    for x in range(len(state)):
        for y in range(len(state[0])):
            # Vị trí của robot đang đứng xem như đã hút sạch bụi
            if state[x][y] != 'X' and state[x][y] != goal_state[x][y]:
                return False

    return True

In [75]:
def find_vacuum_position(state):
    for x in range(len(state)):
        for y in range(len(state[0])):

            if state[x][y] == 'X':
                return x, y
    
    return None

# Tạo ra các trạng thái con hợp lệ và hành động để sinh ra các trạng thái đó
def gen_actions(state):
    m = len(state)
    n = len(state[0])

    x, y = find_vacuum_position(state)

    actions = []

    if x > 0 and state[x-1][y] != 2:
        up_state = [row[:] for row in state]
        clean_action = f" và dọn rác ô [{x-1}][{y}]" if up_state[x-1][y] == 1 else ""

        up_state[x][y], up_state[x-1][y] = 0, 'X'

        actions.append(("Up" + clean_action, up_state))

    if x < m - 1 and state[x+1][y] != 2:
        down_state = [row[:] for row in state]
        clean_action = f" và dọn rác ô [{x+1}][{y}]" if down_state[x+1][y] == 1 else ""

        down_state[x][y], down_state[x+1][y] = 0, 'X'

        actions.append(("Down" + clean_action, down_state))

    if y > 0 and state[x][y-1] != 2:
        left_state = [row[:] for row in state]
        clean_action = f" và dọn rác ô [{x}][{y-1}]" if left_state[x][y-1] == 1 else ""

        left_state[x][y], left_state[x][y-1] = 0, 'X'

        actions.append(("Left" + clean_action, left_state))

    if y < n - 1 and state[x][y+1] != 2:
        right_state = [row[:] for row in state]
        clean_action = f" và dọn rác ô [{x}][{y+1}]" if right_state[x][y+1] == 1 else ""

        right_state[x][y], right_state[x][y+1] = 0, 'X'

        actions.append(("Right" + clean_action, right_state))

    return actions

In [76]:
# Các hàm phụ để so sánh trạng thái đã tồn tại trong frontier và reached hay chưa
def is_state_in_frontier(child_state, frontier):
    for node in frontier:
        if node.state == child_state:
            return True
    
    return False

""" Dùng đối với:
    reached là set, lưu các trạng thái không trùng nhau --> tối ưu tìm kiếm
    state lưu vào phải chuyển thành tuple, vì set chỉ chứa các kiểu dữ liệu bất biến, có thể băm
"""
def is_state_in_reached(child_state, reached):
    return tuple(tuple(row) for row in child_state) in reached

#### Thuật toán BFS cách tiếp cận 1

In [77]:
from collections import deque


def breadth_first_search_version_1(initial, goal_test):
    node = Node(initial, None, None, 0)

    if compare_state(node.state, goal_test):
        return get_solution(node)

    frontier = deque()
    frontier.append(node)

    reached = set()

    while frontier:
        node = frontier.popleft()

        reached.add(tuple(tuple(row) for row in node.state))

        if compare_state(node.state, goal_test):
            return get_solution(node)

        for action, child_state in gen_actions(node.state):

            child = Node(child_state, node, action, node.path_cost + 1)

            if (not is_state_in_frontier(child.state, frontier) and 
                not is_state_in_reached(child.state, reached)):
                frontier.append(child)

    return None

#### Thuật toán BFS cách tiếp cận 2

In [78]:
from collections import deque

def breadth_first_search_version_2(initial, goal_test):
    node = Node(initial, None, None, 0)

    if compare_state(node.state, goal_test):
        return get_solution(node)

    frontier = deque()
    frontier.append(node)

    reached = set()

    while frontier:
        node = frontier.popleft()

        reached.add(tuple(tuple(row) for row in node.state))

        for action, child_state in gen_actions(node.state):

            child = Node(child_state, node, action, node.path_cost + 1)

            if (not is_state_in_frontier(child.state, frontier) and 
                not is_state_in_reached(child.state, reached)):
                
                if compare_state(child.state, goal_test):
                    return get_solution(child)
                
                frontier.append(child)

    return None

#### Thuật toán DFS cách tiếp cận 1

In [79]:
def depth_first_search_version_1(initial, goal_test):
    node = Node(initial, None, None, 0)

    if compare_state(node.state, goal_test):
        return get_solution(node)

    frontier = []
    frontier.append(node)

    reached = set()

    while frontier:
        node = frontier.pop()

        reached.add(tuple(tuple(row) for row in node.state))

        if compare_state(node.state, goal_test):
            return get_solution(node)

        for action, child_state in gen_actions(node.state):

            child = Node(child_state, node, action, node.path_cost + 1)

            if (not is_state_in_frontier(child.state, frontier) and 
                not is_state_in_reached(child.state, reached)):
                frontier.append(child)

    return None

#### Thuật toán DFS cách tiếp cận 2

In [80]:
def depth_first_search_version_2(initial, goal_test):
    node = Node(initial, None, None, 0)

    if compare_state(node.state, goal_test):
        return get_solution(node)

    frontier = []
    frontier.append(node)

    reached = set()

    while frontier:
        node = frontier.pop()

        reached.add(tuple(tuple(row) for row in node.state))

        for action, child_state in gen_actions(node.state):

            child = Node(child_state, node, action, node.path_cost + 1)
        
            if (not is_state_in_frontier(child.state, frontier) and 
                not is_state_in_reached(child.state, reached)):

                if compare_state(child.state, goal_test):
                    return get_solution(child)
            
                frontier.append(child)

    return None

### Thuật toán IDS cách tiếp cận 1

In [81]:
# Hàm phụ để phát hiện chu trình đối với node hiện tại xét
def is_cycle(node: Node):
    cur = node.parent
    
    while cur != None:
        if cur.state == node.state:
            return True
        
        cur = cur.parent
    
    return False

In [82]:
def iterative_deepening_search_version_1(start, goal):
    max_depth = 100
    for depth in range(0, max_depth+1):
        result = depth_limited_search_version_1(start, goal, depth)

        if result != "cutoff":
            if result == "failure":
                return None
            else: 
                return result
    
    return None


def depth_limited_search_version_1(start, goal, l):
    frontier = [Node(start, None, None, 0)]

    result = "failure"

    while frontier:
        node = frontier.pop()
        
        if compare_state(node.state, goal):
            return get_solution(node)
        
        if node.path_cost >= l:
            result = "cutoff"
        
        elif not is_cycle(node):
            for action, child_state in gen_actions(node.state):
                child = Node(child_state, node, action, node.path_cost + 1)

                frontier.append(child)
        
    return result


### Thuật toán IDS cách tiếp cận 2

In [83]:
def iterative_deepening_search_version_2(start, goal):
    max_depth = 100
    for depth in range(0, max_depth+1):
        result = depth_limited_search_version_2(start, goal, depth)

        if result != "cutoff":
            if result == "failure":
                return None
            else: 
                return result
    
    return None


def depth_limited_search_version_2(start, goal, l):
    frontier = [Node(start, None, None, 0)]
    
    if compare_state(frontier[-1].state, goal):
        return get_solution(frontier[-1])

    result = "failure"

    while frontier:
        node = frontier.pop()
        
        if node.path_cost >= l:
            result = "cutoff"
        
        elif not is_cycle(node):
            for action, child_state in gen_actions(node.state):
                child = Node(child_state, node, action, node.path_cost + 1)
                
                if compare_state(child.state, goal):
                    return get_solution(child)

                frontier.append(child)
        
    return result

### Thuật toán UCS

In [84]:
# Cài đặt UCS với path cost (g(n)) bằng g(cha) + số ô còn bụi
def calculate_path_cost(node: Node):
    path_cost = 0 if node.parent == None else node.parent.path_cost

    for i in range(len(node.state)):
        for j in range(len(node.state[0])):
            if node.state[i][j] == 1:
                path_cost += 1
    
    return path_cost

# frontier có dạng priority queue, pop --> lấy phần tử có cost bé nhất
def pop_lowest_cost_ucs_node(frontier):
    lowest_cost_node = min(frontier, key=lambda node: node.path_cost)

    frontier.remove(lowest_cost_node)

    return lowest_cost_node, frontier

# Đối với thuật toán UCS, mỗi phần tử lưu trong reached bao gồm state và cost
def is_state_in_reached_ucs(state, path_cost, reached):
    return (tuple(tuple(row) for row in state), path_cost) in reached
    
    
def uniform_cost_search(initial, goal_test):
    node = Node(initial, None, None, None)
    node.path_cost = calculate_path_cost(node)

    if compare_state(node.state, goal_test):
        return get_solution(node)

    frontier = []
    frontier.append(node)

    reached = set()

    while frontier:
        node, frontier = pop_lowest_cost_ucs_node(frontier)

        reached.add((tuple(tuple(row) for row in node.state), node.path_cost))

        if compare_state(node.state, goal_test):
            return get_solution(node)

        for action, child_state in gen_actions(node.state):

            child = Node(child_state, node, action, None)
            child.path_cost = calculate_path_cost(child)

            if (not is_state_in_frontier(child.state, frontier) and 
                not is_state_in_reached_ucs(child.state, child.path_cost, reached)):
                frontier.append(child)

            elif is_state_in_frontier(child.state, frontier):
                child_in_frontier = list(n for n in frontier if n.state == child.state)[0]

                if child.path_cost < child_in_frontier.path_cost:
                    frontier.remove(child_in_frontier)
                    frontier.append(child)

    return None

### Thuật toán Greedy search  

In [ ]:
def iterative_deepening_search_version_2(start, goal):
    max_depth = 100
    for depth in range(0, max_depth+1):
        result = depth_limited_search_version_2(start, goal, depth)

        if result != "cutoff":
            if result == "failure":
                return None
            else: 
                return result
    
    return None


def depth_limited_search_version_2(start, goal, l):
    frontier = [Node(start, None, None, 0)]
    
    if compare_state(frontier[-1].state, goal):
        return get_solution(frontier[-1])

    result = "failure"

    while frontier:
        node = frontier.pop()
        
        if node.path_cost >= l:
            result = "cutoff"
        
        elif not is_cycle(node):
            for action, child_state in gen_actions(node.state):
                child = Node(child_state, node, action, node.path_cost + 1)
                
                if compare_state(child.state, goal):
                    return get_solution(child)

                frontier.append(child)
        
    return result

### Thuật toán A*  

In [ ]:
# Cài đặt UCS với path cost (g(n)) bằng g(cha) + số ô còn bụi
def calculate_path_cost(node: Node):
    path_cost = 0 if node.parent == None else node.parent.path_cost

    for i in range(len(node.state)):
        for j in range(len(node.state[0])):
            if node.state[i][j] == 1:
                path_cost += 1
    
    return path_cost

# frontier có dạng priority queue, pop --> lấy phần tử có cost bé nhất
def pop_lowest_cost_ucs_node(frontier):
    lowest_cost_node = min(frontier, key=lambda node: node.path_cost)

    frontier.remove(lowest_cost_node)

    return lowest_cost_node, frontier

# Đối với thuật toán UCS, mỗi phần tử lưu trong reached bao gồm state và cost
def is_state_in_reached_ucs(state, path_cost, reached):
    return (tuple(tuple(row) for row in state), path_cost) in reached
    
    
def uniform_cost_search(initial, goal_test):
    node = Node(initial, None, None, None)
    node.path_cost = calculate_path_cost(node)

    if compare_state(node.state, goal_test):
        return get_solution(node)

    frontier = []
    frontier.append(node)

    reached = set()

    while frontier:
        node, frontier = pop_lowest_cost_ucs_node(frontier)

        reached.add((tuple(tuple(row) for row in node.state), node.path_cost))

        if compare_state(node.state, goal_test):
            return get_solution(node)

        for action, child_state in gen_actions(node.state):

            child = Node(child_state, node, action, None)
            child.path_cost = calculate_path_cost(child)

            if (not is_state_in_frontier(child.state, frontier) and 
                not is_state_in_reached_ucs(child.state, child.path_cost, reached)):
                frontier.append(child)

            elif is_state_in_frontier(child.state, frontier):
                child_in_frontier = list(n for n in frontier if n.state == child.state)[0]

                if child.path_cost < child_in_frontier.path_cost:
                    frontier.remove(child_in_frontier)
                    frontier.append(child)

    return None

In [87]:
# Tạo đầu random đầu vào
import random

def gen_random_start_state(row_number=5, col_number=4):
    
    state = [[0] * col_number for _ in range(row_number)]

    # Vị trí robot: ô [0][0]
    state[0][0] = 'X'
 
    # 10% vật cản, 30% ô có rác, 60% ô trống
    for x in range(row_number):
        for y in range(col_number):
            if state[x][y] == 'X':
                continue

            r = random.random()
            if r < 0.1:
                state[x][y] = 2   
            elif r < 0.4:
                state[x][y] = 1  
 
    goal = [[0 if (cell == 'X' or cell == 1) else cell for cell in row] for row in state]
 
    return state, goal

### Giao diện

In [90]:
import tkinter as tk
from tkinter import ttk
import time

ALGORITHM = {
    "BFS version 1": breadth_first_search_version_1,
    "BFS version 2": breadth_first_search_version_2,
    "DFS version 1": depth_first_search_version_1,
    "DFS version 2": depth_first_search_version_2,
    "IDS version 1": iterative_deepening_search_version_1,
    "IDS version 2": iterative_deepening_search_version_2,
    "UCS": uniform_cost_search,
    "Greedy search": greedy_search,
    "A*": a_star_search
}
 
class VacuumGUI:

    def __init__(self, window: tk.Tk):
        self.window = window
        self.window.title("Máy hút bụi")
        self.window.geometry("1280x720+120+50")
        self.window.configure(bg="#EEF5FB")

        self.start =  [['X', 0 , 1 , 1],
                       [ 0 , 2 , 0 , 0],
                       [ 1 , 0 , 1 , 2],
                       [ 2 , 1 , 1 , 1],
                       [ 1 , 0 , 1 , 1]]

        self.goal = [[0, 0, 0, 0],
                     [0, 2, 0, 0],
                     [0, 0, 0, 2],
                     [2, 0, 0, 0],
                     [0, 0, 0, 0]]

        self.animating = False
 
        self.build_layout()
        self.create_grid()
        self.draw_state(self.start)
 

    def build_layout(self):
        
        # Left - Phần setting
        LEFT_FRAME_BG = "#E8F1FD"
        
        self.left_frame = tk.Frame(self.window, bg=LEFT_FRAME_BG, width=200, padx=4, pady=10)
        self.left_frame.pack(side="left", fill="y", padx=8, pady=8)
        self.left_frame.pack_propagate(False)
 
        tk.Label(self.left_frame, text="CÀI ĐẶT", 
                 fg="black", bg=LEFT_FRAME_BG, 
                 font=("Segoe UI", 20, "bold")).pack(anchor="center")
 
        # Chọn thuật toán 
        tk.Label(self.left_frame, text="Thuật toán", 
                 fg="black", bg=LEFT_FRAME_BG, 
                 font=("Segoe UI", 12, "bold")).pack(anchor="w")
 
        self.algorithm_var = tk.StringVar() 
        self.algorithm_box = ttk.Combobox(self.left_frame, 
                                          textvariable=self.algorithm_var, 
                                          values=list(ALGORITHM.keys()), 
                                          state="readonly", width=25)

        self.algorithm_box.current(0) 
        self.algorithm_box.pack(pady=4) 
 
        tk.Button(self.left_frame, text="BẮT ĐẦU", command=self.start_algorithm, fg="white", bg="navy", width=15).pack(pady=5)
        tk.Button(self.left_frame, text="ĐẶT LẠI", command=self.reset, fg="white", bg="saddlebrown", width=15).pack(pady=5)
 
        tk.Frame(self.left_frame, bg="blue", height=1).pack(pady=10,fill="x")

        # Tạo input ngẫu nhiên
        tk.Label(self.left_frame, text="Tạo input ngẫu nhiên", 
                 fg="black", bg=LEFT_FRAME_BG, 
                 font=("Segoe UI", 12, "bold")).pack(anchor="w")

        resize_frame = tk.Frame(self.left_frame, bg=LEFT_FRAME_BG)
        resize_frame.pack(padx=10, pady=8,fill="x")
 
        tk.Label(resize_frame, text="Hàng:", fg="black").grid(row=0, column=0, sticky="w")
        self.row_var = tk.IntVar(value=5)
        tk.Spinbox(resize_frame, from_=2, to=7, textvariable=self.row_var, width=4, fg="black").grid(row=0, column=1, padx=4)
 
        tk.Label(resize_frame, text="Cột:", fg="black").grid(row=1, column=0, sticky="w")
        self.col_var = tk.IntVar(value=4)
        tk.Spinbox(resize_frame, from_=2, to=7, textvariable=self.col_var, width=4, fg="black").grid(row=1, column=1, padx=4)
 
        tk.Button(self.left_frame, text="TẠO", command=self.randomize_map, bg="darkgreen", fg="white", width=15).pack(pady=6)
 
        tk.Frame(self.left_frame, bg="blue", height=1).pack(pady=10,fill="x")

        # Chú thích
        tk.Label(self.left_frame, text="Chú thích", bg=LEFT_FRAME_BG, font=("Segoe UI", 12, "bold")).pack(anchor="w")
        
        tk.Label(self.left_frame, text="🤖: Robot", bg=LEFT_FRAME_BG, fg="black").pack(padx=10, pady=(8, 5), anchor="w")
        tk.Label(self.left_frame, text="🍂: Rác", bg=LEFT_FRAME_BG, fg="black").pack(padx=10, pady=5, anchor="w")
        tk.Label(self.left_frame, text="🧱: Vật cản", bg=LEFT_FRAME_BG, fg="black").pack(padx=10, pady=5, anchor="w")
        tk.Label(self.left_frame, text="Ô trống: Không có rác và vật cản", bg=LEFT_FRAME_BG, fg="black").pack(padx=10, pady=5, anchor="w")
 

        # Center - phía trên: map, phía dưới: kết quả
        self.center_frame = tk.Frame(self.window, bg="#DCEEF8")
        self.center_frame.pack(side="left", fill="both", expand=True, padx=8, pady=8)
 
        self.map_frame = tk.LabelFrame(self.center_frame, text="Sơ đồ", 
                                       bg="#CFE7F5", fg="#0F172A",
                                       font=("Consolas", 13, "bold"),
                                       bd=2, labelanchor="n")
        self.map_frame.pack(fill="both", expand=True, padx=8, pady=4)
 
        self.result_frame = tk.LabelFrame(self.center_frame, text="Kết quả",
                                          bg="#CFE7F5", fg="#0F172A",
                                          font=("Consolas", 13, "bold"),
                                          bd=2, labelanchor="n")
        self.result_frame.pack(fill="x", padx=4, pady=4)
 
        self.result_text = tk.Text(self.result_frame, height=6,
                                   bg="white", fg="#2F4F5F",
                                   state="disabled")
        self.result_text.pack(fill="both", padx=4, pady=4)
 
        # Right - log
        self.log_frame = tk.LabelFrame(self.window, text="Log",
                                       bg="#E6F1F7", fg="#0E0C1B",
                                       font=("Consolas", 11, "bold"),
                                       bd=2, labelanchor="n")
        self.log_frame.pack(side="right", fill="y", padx=8, pady=8)
 
        self.log_scroll = tk.Scrollbar(self.log_frame)
        self.log_scroll.pack(side="right", fill="y")
 
        self.log_text = tk.Text(self.log_frame, width=33, 
                                bg="#E6F1F7", fg="#0F172A",
                                font=("Consolas", 9),
                                yscrollcommand=self.log_scroll.set,
                                state="disabled")
        self.log_text.pack(fill="both", expand=True, padx=4, pady=4)
        self.log_scroll.config(command=self.log_text.yview)
 

    def create_grid(self):
        for widget in self.map_frame.winfo_children():
            widget.destroy()
 
        self.cells = []
        row_number = len(self.start)
        col_number = len(self.start[0])
 
        self.map_frame.grid_columnconfigure(list(range(col_number+2)), weight=1)
        self.map_frame.grid_rowconfigure(list(range(row_number+2)), weight=1)
 
        for i in range(row_number):
            row = []
            for j in range(col_number):
                cell = tk.Label(self.map_frame,
                                width=4, height=2,
                                font=("Segoe UI Emoji", 18))
                cell.grid(row=i+1, column=j+1, padx=3, pady=3, sticky="nsew")
                row.append(cell)
            self.cells.append(row)
 

    def draw_state(self, state):
        cell_dict = {
            "X": {"icon" : "🤖", "color" : "#C7E4F4"},
            0: {"icon" : "", "color" : "#EAF4FA"},
            1: {"icon" : "🍂", "color" : "#DCEAD7"},
            2: {"icon" : "🧱", "color" : "#C8D3DC"}
        }

        for i in range(len(state)):
            for j in range(len(state[0])):
                v = state[i][j]
                self.cells[i][j].config(text=cell_dict.get(v).get("icon"), bg=cell_dict.get(v).get("color"))


    def animate_path(self, path):
        idx = 0
        for node in path:
            if not self.animating:
                return

            self.draw_state(node.state)
            step_label = "Khởi tạo" if node.action is None else node.action
            self.write_log(f"Bước: {idx}\nCost = {node.path_cost}\n{step_label}")
            for row in node.state:
                self.write_log("\t" + " ".join(map(str, row)))
            self.window.update()    
            time.sleep(0.6)

            idx += 1

        self.write_log("Phòng đã được dọn sạch!!!")
        self.animating = False


    def get_result(self, path):
        s = f"Số bước: {len(path)-1}\nChuỗi hành động: "
        for node in path:
            step_action = "Khởi tạo" if node.action is None else node.action
            s += step_action + " → "
        
        s += "Goal"
        return s
        

    def start_algorithm(self):
        if self.animating:
            return
 
        algo_name = self.algorithm_var.get()
        run_algo = ALGORITHM.get(algo_name)
        if run_algo is None:
            return
 
        self.clear_texts()
        self.draw_state(self.start)
 
        self.write_log(f"Thuật toán: {algo_name}")
        self.write_log("Đang tìm đường đi...\n")

        start_time = time.perf_counter()

        path = run_algo(self.start, self.goal)

        end_time = time.perf_counter()
        execution_time = end_time - start_time
 
        if path is None:
            self.write_log("Không tìm thấy đường đi!!!")
            self.write_result("Không tìm thấy đường đi!!!")
            return
        
        self.write_result(f"Thời gian tìm kiếm: {execution_time:.5f} s\n" + self.get_result(path))
        self.write_log("Đã tìm thấy cách di chuyển. \nBắt đầu hành động...\n")
        self.animating = True
        self.animate_path(path)
 

    def reset(self):
        self.animating = False
        self.draw_state(self.start)
        self.clear_texts()
 

    def randomize_map(self):
        self.animating = False
 
        row_number = self.row_var.get()
        col_number = self.col_var.get()
        self.start, self.goal = gen_random_start_state(row_number, col_number)
 
        self.create_grid()
        self.draw_state(self.start)
        self.clear_texts()


    def write_log(self, msg):
        self.log_text.config(state="normal")
        self.log_text.insert(tk.END, msg + "\n")
        self.log_text.see(tk.END)
        self.log_text.config(state="disabled")
 

    def write_result(self, msg):
        self.result_text.config(state="normal")
        self.result_text.insert(tk.END, msg)
        self.result_text.see(tk.END)
        self.result_text.config(state="disabled")
    

    def clear_texts(self):
        self.log_text.config(state="normal")
        self.result_text.config(state="normal")
        self.log_text.delete("1.0", tk.END)
        self.result_text.delete("1.0", tk.END)
        self.log_text.config(state="disabled")
        self.result_text.config(state="disabled")
        

if __name__ == "__main__":
    window = tk.Tk()
    app = VacuumGUI(window)
    window.mainloop()